<a href="https://colab.research.google.com/github/Rj-gif/ayurvedic_healthcare_assistant_2.0/blob/main/Ayurvedic_health_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U langchain-community
!pip install -qU langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.3/554.3 kB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 9.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.0
    Uninstalling langchain-core-1.4.0:
      Successfully uninstalled langchain-core-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.3 MB/s

In [2]:
import os
from google.colab import userdata
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [3]:
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

# ***data_load***

In [4]:
#Directory_loaders
!pip install pypdf
from langchain_community.document_loaders import DirectoryLoader,PyPDFLoader

loader=DirectoryLoader(
    path="/content/drive/MyDrive/data_for_health_assistant",
    glob="*.pdf",
    loader_cls=PyPDFLoader
)
docs=loader.load()



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 11.0 MB/s eta 0:00:00


/tmp/ipykernel_3361/3791696043.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader,PyPDFLoader


In [5]:
print(len(docs))

5076


In [ ]:
print(docs[4830].page_content)

613
  Ka12#31 
 
 Similarly, other above types (gauda) of arista may be prepared in decoction of 
danti and dravanti added with ajagandha or ajasrngi. 
 This acts as simple purgative. 
  Ka12#32 
 
The following madira is useful in K, gulma, mildness of fire and stiffness of sides and 
waist: 
 Madira (wine) prepared of the powder and decoction of danti and dravanti, soup 
of black gram as yeast and water. 
  Ka12#33 
 
 Sauviraka and tusodaka should be prepared of danti and dravanti with decoction 
of ajagandha. 
  Ka12#34 
 
 Their formulations in sura and kampillaka are the same as of lodhra. 
  Ka12#35 
 
 
Summing up– 
 3 preparations in curd etc. 
 5 with priyala etc. 
 3 with meat soups 
 3 in uncting substances 
 6 types of linctus 
 1 powder 
 1 in sugarcane 
 3 in soups of green gram and meat 
 3 in gruel etc. 
 1 in utkarika 
 1 in sweet ball 
 1 in madya (wine) 
 1 in oil with their decoction 
 1 powder 
 1 another sweet ball  
 5 asavas 
 1 each in sauviraka, tusodaka, sur

# ***Chunking***

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter=RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=300,
    separators=["""\n\n\n\n    """," \n", "  \n\n ",    "."]
)

chunks=splitter.split_documents(docs)
print(len(chunks))

7925


In [ ]:
print(chunks[4080].page_content)


print("*"*1000)

print(chunks[4081].page_content)

37. Gokshuradi Guggulu
GOKâURËDI GUGGULU
(AFI, Part-I, 5:3)
Definition:
GokÀur¡di Guggulu Va¶¢  is a preparation made with the ingredients in the Formulation composition  
given below with Guggulu as the basic ingredient.
Formulation composition:
1. GokÀura API Tribulus terrestris Fr. 1.344 kg
2. Jala for decoction     Water 8.064 l
reduced to API 4.032 l
3. Guggulu API Commiphora wightii O.R. 336 g
4. áu¸¶h¢ API Zingiber officinale Rz. 48 g
5. Marica API Piper nigrum Fr. 48 g
6. Pippal¢ API Piper longum Fr. 48 g
7. Har¢tak¢ API Terminalia chebula P. 48 g
8. Bibh¢taka API Terminalia belerica P. 48 g
9. Ëmalak¢ API Emblica officinalis P. 48 g
10. Must¡ API Cyperus rotundus Rz. 48 g
Method of preparation:
Take all the ingredients of the pharmacopoeial quality.
Wash, dry and powder the ingredients number 4 to 10  of the formulation composition separately and  
pass through sieve number 85, weigh them separately in the required quantities and mix.
Crush weighed quantity of Guggulu áuddha. 

# ***Embedding!***

In [7]:
!pip install langchain faiss-cpu langchain-groq tiktoken pypdf langchain-community sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 77.5 MB/s eta 0:00:00


In [8]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [ ]:
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=HuggingFaceEmbeddings()
)

/tmp/ipykernel_2143/3136765807.py:3: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embedding=HuggingFaceEmbeddings()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
vector_store.save_local("/content/drive/MyDrive/ayurvedic_knowledge_base")

In [ ]:
import os
print(os.listdir("/content/drive/MyDrive/ayurvedic_knowledge_base"))
# Must show: ['index.faiss', 'index.pkl']

['index.faiss', 'index.pkl']


for tomorrow
# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Load your saved work ← run THIS tomorrow
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings()

vector_store = FAISS.load_local(
    folder_path="/content/drive/MyDrive/ayurvedic_knowledge_base",
    embeddings=embeddings,
    allow_dangerous_deserialization=True
)
print("✅ Loaded successfully!")

# 3. Directly use it — no rebuilding needed!
results = vector_store.similarity_search("your query")

In [12]:
embeddings = HuggingFaceEmbeddings()

vector_store = FAISS.load_local( folder_path="/content/drive/MyDrive/ayurvedic_knowledge_base",
                                embeddings=embeddings,
                                 allow_dangerous_deserialization=True )
print("✅ Loaded successfully!")

/tmp/ipykernel_3361/841804366.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings()
/tmp/ipykernel_3361/841804366.py:1: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tok

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Loaded successfully!


# ***Retriever***

In [13]:
if vector_store:
    retriever=vector_store.as_retriever(search_type="similarity",search_kwargs={"k":4})
    print("Retriever created successfully.")
else:
    retriever = None
    print("Cannot create retriever: Vector store is not available.")

Retriever created successfully.


# ***Augmentation***

In [14]:
llm=ChatGroq(model="llama-3.3-70b-versatile",temperature=0.2, api_key=GROQ_API_KEY)

In [19]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant.
     Answer only from the provided transcript context.
     If the context is insufficient, just say you don't know.

     Context: {context}"""),
    ("human", "Question: {question}")
])

In [ ]:
question="Is the topic of gastric dicussed there?"
retrieved_docs=retriever.invoke(question)

In [ ]:
retrieved_docs

[Document(id='3a9aac3f-e69d-4544-a687-ac70c60d2f3e', metadata={'producer': 'Acrobat Distiller 5.0.5 (Windows)', 'creator': 'PScript5.dll Version 5.2', 'creationdate': '2003-09-15T18:04:10+00:00', 'moddate': '2003-09-15T14:07:05-04:00', 'author': 'Connie Reavis', 'title': 'Microsoft Word - charakapublish2003.2.volumeI.wrd.doc', 'source': '/content/drive/MyDrive/data_for_health_assistant/Charaka_Samhita_by_Acharya_Charaka.pdf', 'total_pages': 621, 'page': 414, 'page_label': '415'}, page_content='415\nIntake of food during \nindigestion and when \nprevious meal is not \ndigested. \nDryness of mouth, flatulence, colic, \npiercing pain, thirst, lassitude, \nvomiting, diarrhoea, fainting, fever, \ntenesmus, ama visa (food poisoning) \netc.  \nComplete vomiting, rough \nsudation and administration \nof lightening, digestive and \nappetising drugs should be \nprescribed. \nIrregular and \nunwholesome dieting \nloss of desire for food, debility, \nabnormal complexion, itching, exzema, \nburning

In [ ]:
for doc in retrieved_docs:
    print(doc.page_content)
    print("\n---\n")

415
Intake of food during 
indigestion and when 
previous meal is not 
digested. 
Dryness of mouth, flatulence, colic, 
piercing pain, thirst, lassitude, 
vomiting, diarrhoea, fainting, fever, 
tenesmus, ama visa (food poisoning) 
etc.  
Complete vomiting, rough 
sudation and administration 
of lightening, digestive and 
appetising drugs should be 
prescribed. 
Irregular and 
unwholesome dieting 
loss of desire for food, debility, 
abnormal complexion, itching, exzema, 
burning sensation, vomiting, body-
ache, heart-block, dullness, drowsiness, 
excessive sleep, appearance of nodules, 
debility, hematuria, smearing in eyes, 
palate etc. 
Measures for alleviating 
respective dosas should be 
applied. 
Sleeping in day time anorexia, indigestion, loss of digestive 
fire, feeling of wetness; paleness, 
itching, eczema, lassitude and disorders 
caused by vitiated V etc. such as 
grahani, piles etc. appear. 
Smoking, lightening, 
emesis, head-evacuation, 
physical exercise, rough 
diet, use 

In [ ]:
#joining the whole_text
context="\n\n".join(doc.page_content for doc in retrieved_docs)
context

'415\nIntake of food during \nindigestion and when \nprevious meal is not \ndigested. \nDryness of mouth, flatulence, colic, \npiercing pain, thirst, lassitude, \nvomiting, diarrhoea, fainting, fever, \ntenesmus, ama visa (food poisoning) \netc.  \nComplete vomiting, rough \nsudation and administration \nof lightening, digestive and \nappetising drugs should be \nprescribed. \nIrregular and \nunwholesome dieting \nloss of desire for food, debility, \nabnormal complexion, itching, exzema, \nburning sensation, vomiting, body-\nache, heart-block, dullness, drowsiness, \nexcessive sleep, appearance of nodules, \ndebility, hematuria, smearing in eyes, \npalate etc. \nMeasures for alleviating \nrespective dosas should be \napplied. \nSleeping in day time anorexia, indigestion, loss of digestive \nfire, feeling of wetness; paleness, \nitching, eczema, lassitude and disorders \ncaused by vitiated V etc. such as \ngrahani, piles etc. appear. \nSmoking, lightening, \nemesis, head-evacuation, \np

In [ ]:
print(context)

415
Intake of food during 
indigestion and when 
previous meal is not 
digested. 
Dryness of mouth, flatulence, colic, 
piercing pain, thirst, lassitude, 
vomiting, diarrhoea, fainting, fever, 
tenesmus, ama visa (food poisoning) 
etc.  
Complete vomiting, rough 
sudation and administration 
of lightening, digestive and 
appetising drugs should be 
prescribed. 
Irregular and 
unwholesome dieting 
loss of desire for food, debility, 
abnormal complexion, itching, exzema, 
burning sensation, vomiting, body-
ache, heart-block, dullness, drowsiness, 
excessive sleep, appearance of nodules, 
debility, hematuria, smearing in eyes, 
palate etc. 
Measures for alleviating 
respective dosas should be 
applied. 
Sleeping in day time anorexia, indigestion, loss of digestive 
fire, feeling of wetness; paleness, 
itching, eczema, lassitude and disorders 
caused by vitiated V etc. such as 
grahani, piles etc. appear. 
Smoking, lightening, 
emesis, head-evacuation, 
physical exercise, rough 
diet, use 

# ***FULL_CHAIN***

In [15]:
def format_docs(retrieved_docs):
  context="\n\n".join(doc.page_content for doc in retrieved_docs)
  return context

In [16]:
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough

parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [17]:
parser=StrOutputParser()

In [20]:
main_chain=parallel_chain|prompt|llm|parser

In [22]:
question=input("Ask any Question about Ayurveda:")

main_chain.invoke(question)

Ask any Question about Ayurveda:How can I reduce my headache?


'To reduce your headache, you can try the following: \n\n1. Apply a paste of 1/2 tsp. of ginger powder, mixed with water and heated, to the forehead. \n2. Identify the type of headache you have: \n   - For sinus headaches, apply a ginger paste to the forehead and sinuses.\n   - For temporal headaches, drink a tea of cumin and coriander seeds (1/2 tsp. of each in one cup of hot water) and apply sandalwood oil or a sandalwood paste to the temples.\n   - For occipital headaches, take 2 tsps. of flaxseed before sleep in one cup of warm milk and apply a ginger paste behind the ears (mastoid processes).'